# SustAInable — 02: Feature Merge + Engineering
## Neighborhood Heat Illness Risk Prediction
---
**Purpose:** Load the raw synthetic dataset, engineer derived features, 
handle missing values, and export a merged feature matrix ready for modeling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("✅ Libraries loaded.")


## Step 1 — Load Raw Data

In [ ]:
# In production: load from NSSP/BioSense, CDC PLACES, NOAA APIs
# Here: reconstruct synthetic dataset (mirrors NB 01)
N = 1200
np.random.seed(42)

poverty_rate         = np.random.beta(2, 6, N) * 0.6
disability_prev      = np.random.beta(2, 5, N) * 0.45
elderly_pct          = np.random.beta(2, 5, N) * 0.4
ac_access_pct        = np.clip(np.random.beta(5, 2, N), 0.2, 1.0)
unhoused_rate        = np.random.beta(1.5, 10, N) * 0.08
urban_density        = np.random.choice([0,1,2], N, p=[0.2,0.5,0.3])
heat_index_delta     = np.random.normal(3.5, 1.8, N)
tree_canopy_pct      = np.clip(np.random.beta(3, 4, N), 0.01, 0.6)
pct_no_vehicle       = np.random.beta(2, 6, N) * 0.5
median_income_k      = np.random.normal(52, 18, N).clip(15, 120)
region               = np.random.choice(['Northeast','Southeast','Midwest','Southwest','West'],
                                        N, p=[0.22,0.20,0.22,0.18,0.18])

risk_score = (0.25*poverty_rate + 0.20*disability_prev + 0.15*elderly_pct +
              0.15*(1-ac_access_pct) + 0.10*(heat_index_delta/10) +
              0.08*unhoused_rate*5 + 0.07*(1-tree_canopy_pct))

label_prob = 1/(1+np.exp(-8*(risk_score-0.28)))
label_prob = np.clip(label_prob + np.random.normal(0,0.04,N), 0, 1)

df = pd.DataFrame({
    'zcta': [f'{10000+i:05d}' for i in range(N)],
    'region': region, 'urban_density': urban_density,
    'poverty_rate': poverty_rate, 'disability_prevalence': disability_prev,
    'elderly_pct': elderly_pct, 'ac_access_pct': ac_access_pct,
    'unhoused_rate': unhoused_rate, 'heat_index_delta': heat_index_delta,
    'tree_canopy_pct': tree_canopy_pct, 'pct_no_vehicle': pct_no_vehicle,
    'median_income_k': median_income_k,
    'elevated_heat_illness': (label_prob > 0.5).astype(int)
})

print(f"Raw dataset loaded: {df.shape}")
print(df.dtypes)


## Step 2 — Feature Engineering

In [ ]:
# Derived features
df['vulnerability_index'] = (
    df['poverty_rate'] * 0.3 +
    df['disability_prevalence'] * 0.25 +
    df['elderly_pct'] * 0.2 +
    (1 - df['ac_access_pct']) * 0.25
)

df['cooling_deficit']  = (1 - df['ac_access_pct']) * df['heat_index_delta']
df['mobility_barrier'] = df['pct_no_vehicle'] * (1 - df['ac_access_pct'])
df['income_heat_risk']  = df['heat_index_delta'] / (df['median_income_k'] / 50)

# One-hot encode region
df = pd.get_dummies(df, columns=['region'], prefix='region', drop_first=False)

# Urban density: keep as ordinal (0=rural, 1=suburban, 2=urban)
print(f"Feature matrix after engineering: {df.shape}")
print("New features:", ['vulnerability_index','cooling_deficit','mobility_barrier','income_heat_risk'])
df.head(3)


## Step 3 — Missing Value Check

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values.")
print(f"\nFinal shape: {df.shape}")


In [ ]:
eng_features = ['vulnerability_index','cooling_deficit','mobility_barrier','income_heat_risk']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, feat in enumerate(eng_features):
    neg = df[df.elevated_heat_illness==0][feat]
    pos = df[df.elevated_heat_illness==1][feat]
    axes[i].hist(neg, bins=25, alpha=0.6, color='#2196F3', label='No Risk', density=True)
    axes[i].hist(pos, bins=25, alpha=0.6, color='#E53935', label='Elevated Risk', density=True)
    axes[i].set_title(feat.replace('_',' ').title(), fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Engineered Feature Distributions by Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/02_engineered_features.png', dpi=100, bbox_inches='tight')
plt.show()
print("Chart saved.")


In [ ]:
df.to_csv('/tmp/sustainable_features_merged.csv', index=False)
print(f"✅ Feature matrix saved: {df.shape}")
print(f"   Total features (excl. zcta + label): {df.shape[1]-2}")
print(f"   Ready for notebook 03 — label construction + train/test split.")


---
## Notebook Summary
- Engineered 4 derived features: vulnerability index, cooling deficit, mobility barrier, income-heat risk
- One-hot encoded region (5 categories → 5 binary columns)
- No missing values in synthetic data
- **Next:** `03_model_training.ipynb` — train/test split, SMOTE, XGBoost
